# 03 — Rebalancing Strategies

Static optimisation is only half the picture — the other half is
deciding *when* to rebalance. This notebook runs the same universe
through three policies and shows the headline trade-off:

**more frequent rebalances → tighter drift control → higher cost.**

**Reference doc:** [docs/REBALANCING.md](../../../docs/REBALANCING.md).

## 1. Build a 3-year daily returns panel

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from benchmarks.base import generate_synthetic_dataset
from services.rebalancing import RebalancingConfig, run_rebalance_backtest

ds = generate_synthetic_dataset(n_assets=10, n_history=3 * 252, seed=42)
idx = pd.date_range("2022-01-03", periods=ds.n_history, freq="B")
returns_panel = pd.DataFrame(
    ds.daily_returns,
    index=idx,
    columns=[f"A{i:02d}" for i in range(ds.n_assets)],
)
print(f"Panel: {len(returns_panel)} rows  cols={len(returns_panel.columns)}")

## 2. Run three policies

Same optimiser (`markowitz`), same lookback, same costs — only the
rebalancing policy varies. The threshold-drift policy only
rebalances when any weight strays > 5% from target.

In [ ]:
opt_kwargs = {"objective": "markowitz", "weight_max": 0.30}
policies = {
    "monthly":   RebalancingConfig(policy="monthly",   lookback_days=63, cost_linear_bps=5.0),
    "quarterly": RebalancingConfig(policy="quarterly", lookback_days=63, cost_linear_bps=5.0),
    "threshold": RebalancingConfig(
        policy="threshold",
        policy_kwargs={"threshold": 0.05},
        lookback_days=63, cost_linear_bps=5.0,
    ),
}

results = {
    name: run_rebalance_backtest(returns_panel, cfg, opt_kwargs)
    for name, cfg in policies.items()
}

## 3. Summary table

In [ ]:
header = ["policy", "rebals", "gross", "net", "sharpe", "sortino", "mdd", "cost"]
print("  ".join(f"{h:>10}" for h in header))
for name, r in results.items():
    print(
        f"{name:>10}  {r.n_rebalances:>10d}  "
        f"{r.gross_return*100:>9.2f}%  {r.net_return*100:>9.2f}%  "
        f"{r.sharpe:>9.3f}  {r.sortino:>10.3f}  "
        f"{r.max_drawdown*100:>9.2f}%  ${r.cumulative_cost:>8.2f}"
    )

## 4. Equity curves

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
for name, r in results.items():
    ax.plot(pd.to_datetime(r.dates), r.portfolio_values, label=name, linewidth=1.4)
ax.set_ylabel("portfolio value (net of cost)")
ax.set_title("Rebalancing strategies — equity curves")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Cumulative transaction cost

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for name, r in results.items():
    cum_cost = np.cumsum(r.transaction_costs)
    ax.step(
        pd.to_datetime(r.rebalance_dates),
        cum_cost, label=name, where="post", linewidth=1.4,
    )
ax.set_ylabel("cumulative cost ($)")
ax.set_title("Rebalancing cost by policy")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Reading the trade-off

Three things to notice in the table:

1. **Monthly** rebalances most frequently → lowest drift, highest cost.
2. **Threshold-drift** rebalances least → lowest cost, but slightly
   wider drift if the universe is volatile.
3. **Net Sharpe** is the only number that matters for ranking — costs
   eat into the gross return, and the optimal policy depends on the
   asset class and execution cost.

Try changing `cost_linear_bps` to 20 (crypto-ish) and `threshold` to
0.10 to see how the ranking flips when costs dominate.